# RetNet State Extraction - Testing and Verification

This notebook tests the state extraction mechanism for RetNet and compares different extraction methods.

**Goals:**
1. Load the RetNet-2.7B model
2. Compare 3 extraction methods:
   - `extract_states` - Final state only (single forward pass)
   - `extract_states_incremental` - All positions (O(N²) - slow)
   - `extract_incremental_states_single_pass` - All positions (O(N) - efficient)
3. Verify correctness by comparing results
4. Measure and compare performance


In [ ]:
import torch
from typing import Dict, Optional, Tuple
import warnings

from fla.models.retnet import RetNetForCausalLM
from transformers import AutoTokenizer


def load_retnet_model(
    model_name: str = "fla-hub/retnet-2.7B-100B",
    device: Optional[str] = None,
    torch_dtype: torch.dtype = torch.bfloat16,
) -> Tuple[torch.nn.Module, object]:
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        if device == "cpu":
            warnings.warn("CUDA not available. Loading model on CPU. This will be very slow for the 2.7B model.")

    print(f"Loading RetNet model: {model_name}")
    print(f"Device: {device}, dtype: {torch_dtype}")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = RetNetForCausalLM.from_pretrained(model_name, torch_dtype=torch_dtype)
    model = model.to(device)
    model.eval()

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model loaded successfully!")
    print(f"Parameters: {num_params / 1e9:.2f}B")
    print(f"Memory footprint: ~{num_params * 2 / 1e9:.2f} GB (bfloat16)")

    return model, tokenizer


def get_model_config(model: torch.nn.Module) -> Dict:
    config = {}

    if hasattr(model, "config"):
        cfg = model.config
        config["num_layers"] = getattr(cfg, "num_hidden_layers", getattr(cfg, "num_layers", None))
        config["num_heads"] = getattr(cfg, "num_attention_heads", getattr(cfg, "num_heads", None))
        config["hidden_size"] = getattr(cfg, "hidden_size", None)
        config["vocab_size"] = getattr(cfg, "vocab_size", None)
        config["max_seq_len"] = getattr(cfg, "max_position_embeddings", getattr(cfg, "max_seq_len", None))
        config["state_size"] = getattr(cfg, "state_size", config.get("hidden_size"))
        config["decoder_embed_dim"] = getattr(cfg, "decoder_embed_dim", None)
        config["decoder_retention_heads"] = getattr(cfg, "decoder_retention_heads", None)
        config["decoder_layers"] = getattr(cfg, "decoder_layers", None)
        config["full_config"] = cfg
    else:
        warnings.warn("Model does not have a 'config' attribute. Cannot extract architecture details.")

    if config.get("num_layers") is None:
        if hasattr(model, "layers"):
            config["num_layers"] = len(model.layers)
        elif hasattr(model, "decoder") and hasattr(model.decoder, "layers"):
            config["num_layers"] = len(model.decoder.layers)
        elif hasattr(model, "model") and hasattr(model.model, "layers"):
            config["num_layers"] = len(model.model.layers)

    print("\n=== Model Configuration ===")
    for key, value in config.items():
        if key != "full_config":
            print(f"{key}: {value}")
    print("===========================\n")

    return config


def print_model_structure(model: torch.nn.Module, max_depth: int = 3):
    print("\n=== Model Structure ===")

    def print_modules(module, prefix="", depth=0):
        if depth >= max_depth:
            return
        for name, child in module.named_children():
            print(f"{prefix}{name}: {child.__class__.__name__}")
            print_modules(child, prefix=prefix + "  ", depth=depth + 1)

    print_modules(model)
    print("=======================\n")


In [ ]:
import os
import copy
import torch
import torch.nn as nn
from typing import Dict, List, Optional
import numpy as np
import warnings

try:
    import h5py

    HAS_H5PY = True
except ImportError:
    HAS_H5PY = False


class RetNetStateExtractor:
    def __init__(self, model: nn.Module, verbose: bool = True):
        self.model = model
        self.verbose = verbose

    def extract_final_states(self, input_ids: torch.Tensor, use_cache: bool = True) -> Dict[int, torch.Tensor]:
        states = {}

        with torch.no_grad():
            outputs = self.model(input_ids, use_cache=use_cache)

            if outputs.past_key_values is not None:
                for layer_idx, layer_state in enumerate(outputs.past_key_values):
                    if layer_state is not None and "recurrent_state" in layer_state:
                        states[layer_idx] = layer_state["recurrent_state"].detach().cpu()

        if len(states) == 0:
            warnings.warn("No states were captured! Check that use_cache=True and model supports caching.")

        return states

    def extract_incremental_states_dumb_rerunning(
        self,
        input_ids: torch.Tensor,
        layers: Optional[List[int]] = None,
    ) -> Dict[int, Dict[int, torch.Tensor]]:
        seq_len = input_ids.shape[1]
        states_by_position = {}

        if self.verbose:
            print(f"Extracting states incrementally for {seq_len} positions...")

        with torch.no_grad():
            for pos in range(1, seq_len + 1):
                partial_ids = input_ids[:, :pos].contiguous()
                outputs = self.model(partial_ids, use_cache=True)

                position_states = {}
                if outputs.past_key_values is not None:
                    for layer_idx, layer_state in enumerate(outputs.past_key_values):
                        if layers is not None and layer_idx not in layers:
                            continue
                        if layer_state is not None and "recurrent_state" in layer_state:
                            position_states[layer_idx] = layer_state["recurrent_state"].detach().cpu()

                states_by_position[pos] = position_states

        if self.verbose:
            num_layers = len(states_by_position.get(1, {}))
            print(f"Extracted states for {seq_len} positions, {num_layers} layers each")

        return states_by_position

    def extract_incremental_states_single_pass(
        self,
        input_ids: torch.Tensor,
        layers: Optional[List[int]] = None,
    ) -> Dict[int, Dict[int, torch.Tensor]]:
        seq_len = input_ids.shape[1]
        states_by_position = {}

        if self.verbose:
            print(f"Extracting states (single-pass) for {seq_len} positions...")

        with torch.no_grad():
            past_key_values = None

            for pos in range(seq_len):
                if pos == 0:
                    current_ids = input_ids[:, :1]
                else:
                    current_ids = input_ids[:, pos : pos + 1]

                outputs = self.model(
                    current_ids,
                    past_key_values=past_key_values,
                    use_cache=True,
                    sequence_offset=pos,
                    forward_impl="recurrent",
                )

                position_states = {}
                if outputs.past_key_values is not None:
                    for layer_idx, layer_state in enumerate(outputs.past_key_values):
                        if layers is not None and layer_idx not in layers:
                            continue
                        if layer_state is not None and "recurrent_state" in layer_state:
                            position_states[layer_idx] = layer_state["recurrent_state"].detach().cpu().clone()

                states_by_position[pos + 1] = position_states
                past_key_values = copy.deepcopy(outputs.past_key_values)

                if self.verbose and (pos + 1) % 50 == 0:
                    print(f"  Position {pos + 1}/{seq_len}")

        if self.verbose:
            num_layers = len(states_by_position.get(1, {}))
            print(f"Extracted states for {seq_len} positions, {num_layers} layers each")

        return states_by_position


def save_states_to_file(states: Dict, filepath: str):
    _, ext = os.path.splitext(filepath)

    if ext == ".npz":
        states_np = {f"layer_{k}": v.numpy() if isinstance(v, torch.Tensor) else v for k, v in states.items()}
        np.savez(filepath, **states_np)
        print(f"Saved states to {filepath}")

    elif ext == ".h5":
        if not HAS_H5PY:
            raise ImportError("h5py not installed. Install with: pip install h5py")

        with h5py.File(filepath, "w") as f:
            for layer_idx, state in states.items():
                state_np = state.numpy() if isinstance(state, torch.Tensor) else state
                f.create_dataset(f"layer_{layer_idx}", data=state_np)

        print(f"Saved states to {filepath}")

    else:
        raise ValueError(f"Unsupported file extension: {ext}. Use .npz or .h5")


def load_states_from_file(filepath: str) -> Dict[int, np.ndarray]:
    _, ext = os.path.splitext(filepath)

    if ext == ".npz":
        data = np.load(filepath)
        states = {}
        for key in data.keys():
            layer_idx = int(key.split("_")[1])
            states[layer_idx] = data[key]
        print(f"Loaded states from {filepath}: {len(states)} layers")
        return states

    elif ext == ".h5":
        if not HAS_H5PY:
            raise ImportError("h5py not installed. Install with: pip install h5py")

        states = {}
        with h5py.File(filepath, "r") as f:
            for key in f.keys():
                layer_idx = int(key.split("_")[1])
                states[layer_idx] = f[key][:]

        print(f"Loaded states from {filepath}: {len(states)} layers")
        return states

    else:
        raise ValueError(f"Unsupported file extension: {ext}. Use .npz or .h5")


## 1. Load Model and Configuration


In [ ]:
# Check if CUDA is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cpu":
    print("WARNING: Running on CPU. This will be very slow for 2.7B model.")
    print("Consider running on a GPU or using a smaller model for testing.")


In [ ]:
# Load the model
print("Loading RetNet-2.7B model...")
model, tokenizer = load_retnet_model(
    model_name="fla-hub/retnet-2.7B-100B",
    device=device,
    torch_dtype=torch.bfloat16
)


In [ ]:
# Get model configuration
config = get_model_config(model)

print("\n=== Key Configuration ===")
print(f"Number of layers: {config.get('num_layers', 'Unknown')}")
print(f"Number of heads: {config.get('num_heads', 'Unknown')}")
print(f"Hidden size: {config.get('hidden_size', 'Unknown')}")
print(f"Vocabulary size: {config.get('vocab_size', 'Unknown')}")
print(f"Max sequence length: {config.get('max_seq_len', 'Unknown')}")


In [ ]:
# Print model structure to understand layer organization
print_model_structure(model, max_depth=3)


## 2. Initialize State Extractor and Prepare Test Input


In [ ]:
extractor = RetNetStateExtractor(model, verbose=True)

# Prepare test input
test_text = "The quick brown fox jumps over the lazy dog."
print(f"Input text: '{test_text}'")

inputs = tokenizer(test_text, return_tensors="pt")
input_ids = inputs.input_ids.to(device)

print(f"Token IDs shape: {input_ids.shape}")
print(f"Tokens: {tokenizer.convert_ids_to_tokens(input_ids[0])}")
print(f"\nState extractor ready")


In [ ]:
# Debug: Inspect cache structure directly
with torch.no_grad():
    outputs = model(input_ids, use_cache=True)

print(f"Outputs type: {type(outputs)}")
print(f"past_key_values type: {type(outputs.past_key_values)}")
print(f"Number of layers: {len(outputs.past_key_values)}")

print(f"\nFirst layer cache:")
first_layer = outputs.past_key_values[0]
print(f"Type: {type(first_layer)}")
print(f"Keys: {first_layer.keys() if isinstance(first_layer, dict) else 'Not a dict'}")

if "recurrent_state" in first_layer:
    print(f"\nrecurrent_state shape: {first_layer['recurrent_state'].shape}")
    print(f"recurrent_state dtype: {first_layer['recurrent_state'].dtype}")
    
print(f"\nAll available keys in first layer:")
for key, value in first_layer.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: shape={value.shape}, dtype={value.dtype}")

## 3. Compare Extraction Methods

We compare three extraction methods:
1. **`extract_states`** - Gets only the final state (single forward pass)
2. **`extract_states_incremental`** - Gets all intermediate states (O(N²) - runs full prefix for each position)
3. **`extract_incremental_states_single_pass`** - Gets all intermediate states (O(N) - efficient incremental processing)


In [ ]:
# Method 1: extract_final_states - Final state only (single forward pass)
print("=" * 60)
print("METHOD 1: extract_final_states (final state only)")
print("=" * 60)

start_time = time.time()
final_states = extractor.extract_final_states(input_ids)
time_method1 = time.time() - start_time

print(f"\nTime: {time_method1:.4f}s")
print(f"Number of layers: {len(final_states)}")
if final_states:
    first_layer_state = final_states[0]
    print(f"State shape per layer: {first_layer_state.shape}")


In [ ]:
# Method 2: extract_incremental_states_dumb_rerunning - All positions (O(N²) - slow)
print("=" * 60)
print("METHOD 2: extract_incremental_states_dumb_rerunning (O(N²) - slow)")
print("=" * 60)

start_time = time.time()
incremental_states = extractor.extract_incremental_states_dumb_rerunning(input_ids)
time_method2 = time.time() - start_time

print(f"\nTime: {time_method2:.4f}s")
print(f"Number of positions: {len(incremental_states)}")
first_pos_states = incremental_states[1]
print(f"Number of layers per position: {len(first_pos_states)}")
print(f"State shape at each position: {first_pos_states[0].shape}")

In [ ]:
# Method 3: extract_incremental_states_single_pass - All positions (O(N) - efficient)
print("=" * 60)
print("METHOD 3: extract_incremental_states_single_pass (O(N) - efficient)")
print("=" * 60)

start_time = time.time()
single_pass_states = extractor.extract_incremental_states_single_pass(input_ids)
time_method3 = time.time() - start_time

print(f"\nTime: {time_method3:.4f}s")
print(f"Number of positions: {len(single_pass_states)}")
first_pos_states_sp = single_pass_states[1]
print(f"Number of layers per position: {len(first_pos_states_sp)}")
print(f"State shape at each position: {first_pos_states_sp[0].shape}")

## 4. Compare Results and Verify Correctness

This section verifies that the efficient method produces the same results as the slow method.


In [ ]:
# Compare timing
print("=" * 60)
print("TIMING COMPARISON")
print("=" * 60)
seq_len = input_ids.shape[1]
print(f"\nSequence length: {seq_len} tokens")
print(f"\nMethod 1 (final state only):    {time_method1:.4f}s")
print(f"Method 2 (incremental, O(N²)):  {time_method2:.4f}s")
print(f"Method 3 (single-pass, O(N)):   {time_method3:.4f}s")
print(f"\nSpeedup (Method 3 vs Method 2): {time_method2 / time_method3:.2f}x")


In [ ]:
# Shape verification: check that all extraction methods produce same shapes
print("=" * 60)
print("SHAPE VERIFICATION")
print("=" * 60)

seq_len = input_ids.shape[1]

print("\n1. Comparing shapes: Method 1 (final) vs Method 2 (last position):")
shapes_match_1_vs_2 = True
for layer_idx in final_states.keys():
    state_m1 = final_states[layer_idx]
    state_m2 = incremental_states[seq_len][layer_idx]
    if state_m1.shape != state_m2.shape:
        shapes_match_1_vs_2 = False
        print(f"   Layer {layer_idx}: MISMATCH - {state_m1.shape} vs {state_m2.shape}")
print(f"   All shapes match: {shapes_match_1_vs_2}")

print("\n2. Comparing shapes: Method 1 (final) vs Method 3 (last position):")
shapes_match_1_vs_3 = True
for layer_idx in final_states.keys():
    state_m1 = final_states[layer_idx]
    state_m3 = single_pass_states[seq_len][layer_idx]
    if state_m1.shape != state_m3.shape:
        shapes_match_1_vs_3 = False
        print(f"   Layer {layer_idx}: MISMATCH - {state_m1.shape} vs {state_m3.shape}")
print(f"   All shapes match: {shapes_match_1_vs_3}")

print("\n3. Comparing shapes: Method 2 vs Method 3 (all positions):")
shapes_match_2_vs_3 = True
shape_mismatches = []
for pos in incremental_states.keys():
    for layer_idx in incremental_states[pos].keys():
        state_m2 = incremental_states[pos][layer_idx]
        state_m3 = single_pass_states[pos][layer_idx]
        if state_m2.shape != state_m3.shape:
            shapes_match_2_vs_3 = False
            shape_mismatches.append((pos, layer_idx, state_m2.shape, state_m3.shape))

if shape_mismatches:
    print(f"   Found {len(shape_mismatches)} shape mismatches:")
    for pos, layer_idx, shape_m2, shape_m3 in shape_mismatches[:5]:
        print(f"     Position {pos}, Layer {layer_idx}: {shape_m2} vs {shape_m3}")
else:
    print(f"   All shapes match: {shapes_match_2_vs_3}")


In [ ]:
# Value verification: compare final states from all methods
print("=" * 60)
print("VALUE VERIFICATION")
print("=" * 60)

seq_len = input_ids.shape[1]

print("\n1. Comparing values: Method 1 (final) vs Method 2 (last position):")
all_match_1_vs_2 = True
for layer_idx in final_states.keys():
    state_m1 = final_states[layer_idx]
    state_m2 = incremental_states[seq_len][layer_idx]
    is_close = torch.allclose(state_m1, state_m2, rtol=1e-4, atol=1e-6)
    if not is_close:
        all_match_1_vs_2 = False
        max_diff = (state_m1 - state_m2).abs().max().item()
        print(f"   Layer {layer_idx}: MISMATCH (max diff: {max_diff:.2e})")
print(f"   All values match: {all_match_1_vs_2}")

print("\n2. Comparing values: Method 1 (final) vs Method 3 (last position):")
all_match_1_vs_3 = True
for layer_idx in final_states.keys():
    state_m1 = final_states[layer_idx]
    state_m3 = single_pass_states[seq_len][layer_idx]
    is_close = torch.allclose(state_m1, state_m3, rtol=1e-4, atol=1e-6)
    if not is_close:
        all_match_1_vs_3 = False
        max_diff = (state_m1 - state_m3).abs().max().item()
        print(f"   Layer {layer_idx}: MISMATCH (max diff: {max_diff:.2e})")
print(f"   All values match: {all_match_1_vs_3}")

print("\n3. Comparing values: Method 2 vs Method 3 (all positions):")
all_match_2_vs_3 = True
mismatches = []
for pos in incremental_states.keys():
    for layer_idx in incremental_states[pos].keys():
        state_m2 = incremental_states[pos][layer_idx]
        state_m3 = single_pass_states[pos][layer_idx]
        is_close = torch.allclose(state_m2, state_m3, rtol=1e-4, atol=1e-6)
        if not is_close:
            all_match_2_vs_3 = False
            max_diff = (state_m2 - state_m3).abs().max().item()
            mismatches.append((pos, layer_idx, max_diff))

if mismatches:
    print(f"   Found {len(mismatches)} value mismatches:")
    for pos, layer_idx, diff in mismatches[:5]:
        print(f"     Position {pos}, Layer {layer_idx}: max diff = {diff:.2e}")
else:
    print(f"   All values match: {all_match_2_vs_3}")


In [ ]:
# Visualize timing comparison
fig, ax = plt.subplots(figsize=(10, 6))

methods = ['Method 1\n(final only)', 'Method 2\n(incremental O(N²))', 'Method 3\n(single-pass O(N))']
times = [time_method1, time_method2, time_method3]
colors = ['#2ecc71', '#e74c3c', '#3498db']

bars = ax.bar(methods, times, color=colors, edgecolor='black', linewidth=1.5)

for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{t:.3f}s', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_title(f'State Extraction Methods Comparison\n(Sequence length: {seq_len} tokens)', fontsize=14)
ax.set_ylim(0, max(times) * 1.2)

plt.tight_layout()
plt.show()

print(f"\nSummary:")
print(f"- For final state only: Use Method 1 (fastest)")
print(f"- For all intermediate states: Use Method 3 (single-pass)")
print(f"- Method 3 is {time_method2/time_method3:.1f}x faster than Method 2 for this sequence")
